# 04 — H1: Interrupted Time Series Analysis (Ukraine, February 2022)

**Hypothesis H1:** The time series of search activity for psychological help in Ukraine shows a statistically significant break (level and/or slope) in February 2022.

**Method:** Segmented OLS regression (Interrupted Time Series).

**OSF pre-registration:** https://osf.io/7986u

---

**Model specification (per OSF protocol):**
```
Y(t) = β0 + β1·time + β2·intervention + β3·time_after + ε
```
where:
- `time` = integer months from study start (Jan 2018 = 0)
- `intervention` = 0 before Feb 2022, 1 from Feb 2022 onwards
- `time_after` = 0 before Feb 2022, then 1, 2, 3... months after
- `β1` = pre-intervention trend
- `β2` = level change at February 2022
- `β3` = slope change after February 2022

**Autocorrelation check:** Durbin-Watson test.  
If significant → Newey-West HAC standard errors applied.

**Analysis run separately for each keyword category.**  
**Bonferroni correction:** α = 0.05/3 = 0.017

## 0. Setup

In [ ]:
import json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import statsmodels.api as sm
from statsmodels.stats.stattools import durbin_watson  # autocorrelation test
from scipy import stats
from pathlib import Path

warnings.filterwarnings('ignore')

DATA_DIR   = Path('../data')
OUTPUT_DIR = Path('../results/H1')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FOCAL_COUNTRY   = 'UA'
BREAKPOINT_DATE = pd.Timestamp('2022-02-01')   # Feb 2022 — Russia's full-scale invasion
CATEGORIES      = ['general_help_seeking', 'evidence_based_treatments', 'specific_conditions']
ALPHA           = 0.05
ALPHA_BONF      = ALPHA / 3   # Bonferroni: 3 categories tested

# Durbin-Watson: values outside [1.5, 2.5] indicate autocorrelation
# DW ~ 2 = no autocorrelation; DW < 1.5 = positive; DW > 2.5 = negative
DW_LOWER = 1.5
DW_UPPER = 2.5

print(f'Focal country:   {FOCAL_COUNTRY}')
print(f'Breakpoint:      {BREAKPOINT_DATE.date()}')
print(f'Categories:      {CATEGORIES}')
print(f'Alpha (Bonf.):   {ALPHA_BONF:.3f}')


## 1. Load Ukraine data

In [ ]:
def load_country_category(country_dir: Path, category: str) -> pd.Series | None:
    """Same as H2a/H2b: load CSVs, clean, return monthly mean RSV."""
    csv_files = sorted(country_dir.glob(f'{category}__*__chunk*.csv'))
    if not csv_files:
        return None
    frames = []
    for f in csv_files:
        try:
            df = pd.read_csv(f, parse_dates=['date'], index_col='date')
            if 'isPartial' in df.columns:
                df = df[df['isPartial'] == False].drop(columns=['isPartial'])
            df = df.replace(0, np.nan)
            all_nan_cols = df.columns[df.isna().all()]
            df = df.drop(columns=all_nan_cols)
            if df.empty or df.shape[1] == 0:
                continue
            frames.append(df)
        except Exception as e:
            print(f'  Warning: {f.name}: {e}')
    if not frames:
        return None
    return pd.concat(frames, axis=1).mean(axis=1)


raw_folders = sorted(DATA_DIR.glob('raw_*'))
if not raw_folders:
    raise FileNotFoundError(f'No raw_* folders found in {DATA_DIR}')
LATEST = raw_folders[-1]
UA_DIR = LATEST / FOCAL_COUNTRY

if not UA_DIR.exists():
    raise FileNotFoundError(f'Ukraine folder not found: {UA_DIR}')

print(f'Data folder: {LATEST.name}')
print(f'Ukraine dir: {UA_DIR}')

# Load all three categories — ITS runs separately for each
ua_data = {}
for cat in CATEGORIES:
    series = load_country_category(UA_DIR, cat)
    if series is None:
        print(f'  WARNING: No data for category "{cat}"')
    else:
        ua_data[cat] = series
        print(f'  {cat}: {len(series)} months loaded')


## 2. Build ITS variables

Per OSF protocol:
- `time` = months from Jan 2018 (0, 1, 2, ...)
- `intervention` = 0 before Feb 2022, 1 from Feb 2022
- `time_after` = 0 before Feb 2022, then 1, 2, 3...

In [ ]:
def build_its_variables(series: pd.Series, breakpoint: pd.Timestamp) -> pd.DataFrame:
    """
    Build ITS design matrix for segmented regression.

    Model: Y(t) = β0 + β1*time + β2*intervention + β3*time_after + ε
    - time:         0, 1, 2, ..., N-1  (months from study start)
    - intervention: 0 before breakpoint, 1 from breakpoint onwards
    - time_after:   0 before breakpoint, then 1, 2, 3... months after
    """
    df = pd.DataFrame({'Y': series})
    df.index = pd.to_datetime(df.index)
    df = df.sort_index()
    n = len(df)

    df['time'] = np.arange(n)

    # Step dummy: 0 → 1 at breakpoint
    df['intervention'] = (df.index >= breakpoint).astype(int)

    # Ramp variable: counts months elapsed since breakpoint (0 before)
    # Used to model slope change — distinct from level change (intervention)
    bp_idx = df.index.get_loc(breakpoint) if breakpoint in df.index else \
             np.searchsorted(df.index, breakpoint)
    time_after = np.zeros(n, dtype=int)
    after_count = n - bp_idx
    time_after[bp_idx:] = np.arange(1, after_count + 1)
    df['time_after'] = time_after

    return df


cat_preview = list(ua_data.keys())[0]
df_preview = build_its_variables(ua_data[cat_preview], BREAKPOINT_DATE)

print('ITS variables preview (around breakpoint):')
mask_bp = (df_preview.index >= pd.Timestamp('2022-01-01')) & \
          (df_preview.index <= pd.Timestamp('2022-04-01'))
print(df_preview[mask_bp].to_string())
print(f'\nPre-intervention periods:  {(df_preview["intervention"]==0).sum()}')
print(f'Post-intervention periods: {(df_preview["intervention"]==1).sum()}')


## 3. ITS model — run per category

In [ ]:
def run_its_model(df: pd.DataFrame, alpha_bonf: float, dw_lower: float, dw_upper: float):
    """
    Fit ITS segmented regression with optional Newey-West HAC correction.

    Steps:
    1. Fit OLS with 4 predictors: const, time, intervention, time_after
    2. Durbin-Watson test on residuals → detect autocorrelation
    3. If autocorrelation detected → refit with HAC standard errors (maxlags=3)
    4. Return params, p-values, CIs, and significance flags
    """
    mask   = ~df['Y'].isna()
    df_fit = df[mask].copy()
    y = df_fit['Y'].values
    X = sm.add_constant(df_fit[['time', 'intervention', 'time_after']])

    # Step 1: plain OLS
    ols = sm.OLS(y, X).fit()

    # Step 2: Durbin-Watson statistic (range 0–4; ~2 = no autocorrelation)
    dw_stat = durbin_watson(ols.resid)
    autocorr_detected = not (dw_lower <= dw_stat <= dw_upper)

    # Step 3: if autocorrelation → Newey-West HAC corrects standard errors
    # maxlags=3: accounts for correlations up to 3 months apart
    # Point estimates (betas) stay the same; only SEs/p-values change
    if autocorr_detected:
        model_primary = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': 3})
        method = 'OLS + Newey-West HAC'
    else:
        model_primary = ols
        method = 'OLS'

    ci = model_primary.conf_int()

    return {
        'n': int(len(y)),
        'method': method,
        'autocorrelation_detected': autocorr_detected,
        'durbin_watson': round(float(dw_stat), 4),
        'params': {
            'beta0': round(float(model_primary.params['const']), 4),
            'beta1_trend': round(float(model_primary.params['time']), 4),
            'beta2_level': round(float(model_primary.params['intervention']), 4),
            'beta3_slope': round(float(model_primary.params['time_after']), 4),
        },
        'pvalues': {
            'beta1_trend': round(float(model_primary.pvalues['time']), 4),
            'beta2_level': round(float(model_primary.pvalues['intervention']), 4),
            'beta3_slope': round(float(model_primary.pvalues['time_after']), 4),
        },
        'ci': {
            'beta2_level': [round(float(ci.loc['intervention', 0]), 4),
                            round(float(ci.loc['intervention', 1]), 4)],
            'beta3_slope': [round(float(ci.loc['time_after', 0]), 4),
                            round(float(ci.loc['time_after', 1]), 4)],
        },
        'r_squared': round(float(ols.rsquared), 4),
        # H1 confirmed if EITHER level OR slope change is significant
        'significant_level_change': bool(model_primary.pvalues['intervention'] < alpha_bonf),
        'significant_slope_change': bool(model_primary.pvalues['time_after'] < alpha_bonf),
        '_model': model_primary,
        '_df_fit': df_fit,
    }


its_results = {}

for cat in ua_data:
    df_its = build_its_variables(ua_data[cat], BREAKPOINT_DATE)
    result = run_its_model(df_its, ALPHA_BONF, DW_LOWER, DW_UPPER)
    its_results[cat] = result

    print(f'\n{"═"*55}')
    print(f'Category: {cat}')
    print(f'Method:   {result["method"]}')
    print(f'DW stat:  {result["durbin_watson"]}  '
          f'(autocorrelation: {"YES" if result["autocorrelation_detected"] else "no"})')
    print(f'R²:       {result["r_squared"]}')
    print()
    p  = result['params']
    pv = result['pvalues']
    print(f'  β1 (pre-trend):   {p["beta1_trend"]:>8.4f}  p={pv["beta1_trend"]:.4f}')
    print(f'  β2 (level Δ):     {p["beta2_level"]:>8.4f}  p={pv["beta2_level"]:.4f}  '
          f'→ {"SIGNIFICANT" if result["significant_level_change"] else "ns"}')
    print(f'  β3 (slope Δ):     {p["beta3_slope"]:>8.4f}  p={pv["beta3_slope"]:.4f}  '
          f'→ {"SIGNIFICANT" if result["significant_slope_change"] else "ns"}')
    print()
    h1 = result['significant_level_change'] or result['significant_slope_change']
    print(f'  H1 ({cat}): {"SUPPORTED" if h1 else "NOT SUPPORTED"}')


## 4. Visualization

In [ ]:
n_cats = len(its_results)
fig, axes = plt.subplots(n_cats, 1, figsize=(14, 5 * n_cats))
if n_cats == 1:
    axes = [axes]

fig.suptitle('H1 — Interrupted Time Series Analysis\nUkraine, breakpoint: February 2022',
             fontsize=13, fontweight='bold')

for ax, (cat, res) in zip(axes, its_results.items()):
    df_fit = res['_df_fit']
    model  = res['_model']

    # Observed data
    ax.plot(df_fit.index, df_fit['Y'],
            color='steelblue', linewidth=1.5, alpha=0.8, label='Observed RSV')

    # ITS fitted line (includes β2 and β3 effects)
    ax.plot(df_fit.index, model.fittedvalues,
            color='darkorange', linewidth=2, linestyle='--', label='ITS fitted')

    # Counterfactual: what would have happened WITHOUT the intervention
    # = β0 + β1*time only (intervention=0, time_after=0 for ALL months)
    pre_mask = df_fit['intervention'] == 0
    if pre_mask.sum() >= 2:
        X_counter = df_fit[['time']].copy()
        X_counter.insert(0, 'const', 1)
        X_counter['intervention'] = 0
        X_counter['time_after']   = 0
        counterfactual = model.predict(X_counter)
        ax.plot(df_fit.index, counterfactual,
                color='grey', linewidth=1.5, linestyle=':', label='Counterfactual')

    ax.axvline(BREAKPOINT_DATE, color='red', linestyle='--',
               linewidth=1.5, label='Feb 2022')

    p  = res['params']
    pv = res['pvalues']
    txt = (f"β2 (level Δ) = {p['beta2_level']:.2f}, p={pv['beta2_level']:.3f}\n"
           f"β3 (slope Δ) = {p['beta3_slope']:.2f}, p={pv['beta3_slope']:.3f}\n"
           f"Method: {res['method']}  |  R²={res['r_squared']}")
    ax.text(0.02, 0.97, txt, transform=ax.transAxes,
            fontsize=8, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

    ax.set_title(f'Category: {cat}', fontsize=11)
    ax.set_xlabel('Date')
    ax.set_ylabel('Mean RSV (0–100)')
    ax.legend(fontsize=8, loc='upper left')
    ax.grid(alpha=0.3)
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'H1_ITS_Ukraine.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Figure saved to {OUTPUT_DIR / "H1_ITS_Ukraine.png"}')


## 5. Sensitivity analysis

Per OSF protocol: ITS model re-run with alternative breakpoints (March 2022, April 2022).

In [ ]:
ALT_BREAKPOINTS = [
    pd.Timestamp('2022-01-01'),   # Google artifact control: GT change effective Jan 1, 2022
    pd.Timestamp('2022-03-01'),   # robustness: delayed behavioral response
    pd.Timestamp('2022-04-01'),   # robustness: further delayed response
]

sensitivity = {}

for bp in ALT_BREAKPOINTS:
    sensitivity[str(bp.date())] = {}
    for cat in ua_data:
        df_its = build_its_variables(ua_data[cat], bp)
        res    = run_its_model(df_its, ALPHA_BONF, DW_LOWER, DW_UPPER)
        sensitivity[str(bp.date())][cat] = {
            'beta2_level': res['params']['beta2_level'],
            'p_beta2':     res['pvalues']['beta2_level'],
            'beta3_slope': res['params']['beta3_slope'],
            'p_beta3':     res['pvalues']['beta3_slope'],
            'significant': res['significant_level_change'] or res['significant_slope_change'],
        }

print('Sensitivity Analysis — Alternative Breakpoints')
print('=' * 60)
print()
print('NOTE: January 2022 breakpoint serves as a Google Trends artifact control.')
print('      If β2 is NOT significant at Jan 2022 but IS significant at Feb 2022,')
print('      this supports a genuine war effect rather than a data collection artifact.')
print()
for bp_date, cats in sensitivity.items():
    label = ' ← [ARTIFACT CONTROL]' if bp_date == '2022-01-01' else ''
    print(f'Breakpoint: {bp_date}{label}')
    for cat, r in cats.items():
        sig = 'SUPPORTED' if r['significant'] else 'not supported'
        print(f'  {cat[:30]:30s}  β2={r["beta2_level"]:7.3f} p={r["p_beta2"]:.3f}  '
              f'β3={r["beta3_slope"]:7.3f} p={r["p_beta3"]:.3f}  → {sig}')
    print()


## 6. Export results

In [ ]:
# Strip non-serializable objects (_model, _df_fit) before JSON export
export = {}
for cat, res in its_results.items():
    r = {k: v for k, v in res.items() if not k.startswith('_')}
    export[cat] = r

full_results = {
    'hypothesis':           'H1',
    'country':              FOCAL_COUNTRY,
    'breakpoint':           str(BREAKPOINT_DATE.date()),
    'alpha_bonferroni':     ALPHA_BONF,
    'categories':           export,
    'sensitivity_analysis': sensitivity,
}

results_path = OUTPUT_DIR / 'H1_results.json'
with open(results_path, 'w', encoding='utf-8') as f:
    json.dump(full_results, f, indent=2, ensure_ascii=False)

print('Results saved:')
print(f'  {results_path}')
print()
print('═' * 55)
print('SUMMARY — H1')
print('═' * 55)
for cat, res in export.items():
    h1 = res['significant_level_change'] or res['significant_slope_change']
    print(f'{cat}:')
    print(f'  Method: {res["method"]}')
    print(f'  β2 (level Δ): {res["params"]["beta2_level"]}  p={res["pvalues"]["beta2_level"]}')
    print(f'  β3 (slope Δ): {res["params"]["beta3_slope"]}  p={res["pvalues"]["beta3_slope"]}')
    print(f'  H1: {"SUPPORTED" if h1 else "NOT SUPPORTED"}')
    print()
